In [1]:
# ============================================================
# IMPORTAÇÕES — VIEWS MYSQL / POWER BI
# ============================================================

import pandas as pd

from getpass import getpass

from sqlalchemy import (
    create_engine,
    text,
    inspect
)

from sqlalchemy.engine import URL


print("=" * 70)
print("NOTEBOOK 13 — VIEWS MYSQL / POWER BI")
print("=" * 70)

NOTEBOOK 13 — VIEWS MYSQL / POWER BI


In [4]:
# ============================================================
# CONEXÃO MYSQL
# ============================================================

import getpass as gp

from sqlalchemy import (
    create_engine,
    text
)

from sqlalchemy.engine import URL


USUARIO_MYSQL = input(
    "Usuário MySQL: "
)


SENHA_MYSQL = gp.getpass(
    "Senha MySQL: "
)


url_mysql = URL.create(
    drivername="mysql+pymysql",
    username=USUARIO_MYSQL,
    password=SENHA_MYSQL,
    host="localhost",
    port=3306,
    database="agroesg_analytics",
    query={
        "charset": "utf8mb4"
    }
)


engine = create_engine(
    url_mysql,
    pool_pre_ping=True
)


with engine.connect() as conexao:

    resultado = (
        conexao.execute(
            text(
                """
                SELECT
                    VERSION() AS versao,
                    DATABASE() AS banco;
                """
            )
        )
        .mappings()
        .one()
    )


print("=" * 70)
print("CONEXÃO MYSQL")
print("=" * 70)


print(
    "\nMySQL:",
    resultado["versao"]
)


print(
    "Banco:",
    resultado["banco"]
)

Usuário MySQL:  root
Senha MySQL:  ········


CONEXÃO MYSQL

MySQL: 8.0.46
Banco: agroesg_analytics


In [5]:
# ============================================================
# VIEW — MUNICÍPIOS DE PRIORIDADE ESTRATÉGICA
# ============================================================

sql_view_prioridade = """
CREATE OR REPLACE VIEW vw_prioridade_estrategica AS

SELECT
    codigo_ibge,
    municipio,
    uf,
    regiao,

    producao_media_t,
    area_colhida_media_ha,
    rendimento_medio_kg_ha,

    carbono_solo_2024_t_ha,
    cobertura_natural_2024_pct,
    soja_mapbiomas_2024_pct,

    percentual_conversao_para_soja_pct,
    taxa_emissao_co2_t_ha_ano,

    score_relevancia_produtiva,
    score_pressao_agroambiental,

    quadrante_priorizacao,
    faixa_confianca_modelo,

    quantidade_cenarios_estrategico,
    faixa_robustez_priorizacao,

    flag_prioridade_estrategica_base,
    flag_prioridade_robusta_3_ou_4_cenarios

FROM mart_priorizacao_municipal

WHERE
    flag_prioridade_estrategica_base = TRUE;
"""


with engine.begin() as conexao:

    conexao.execute(
        text(
            sql_view_prioridade
        )
    )


print("=" * 70)
print("VIEW CRIADA")
print("=" * 70)

print(
    "\nvw_prioridade_estrategica"
)

VIEW CRIADA

vw_prioridade_estrategica


In [6]:
# ============================================================
# VALIDAÇÃO — VIEW PRIORIDADE ESTRATÉGICA
# ============================================================

with engine.connect() as conexao:

    total_prioridade = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM vw_prioridade_estrategica;
                """
            )
        )
        .scalar()
    )


    municipios_prioridade = (
        conexao.execute(
            text(
                """
                SELECT COUNT(DISTINCT codigo_ibge)
                FROM vw_prioridade_estrategica;
                """
            )
        )
        .scalar()
    )


    duplicatas_prioridade = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        COUNT(*) AS quantidade
                    FROM vw_prioridade_estrategica
                    GROUP BY codigo_ibge
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


print("=" * 70)
print("VALIDAÇÃO — VW_PRIORIDADE_ESTRATEGICA")
print("=" * 70)


print(
    "\nRegistros:",
    total_prioridade
)


print(
    "Municípios distintos:",
    municipios_prioridade
)


print(
    "Duplicatas:",
    duplicatas_prioridade
)


# ------------------------------------------------------------
# Distribuição por região e UF
# ------------------------------------------------------------

distribuicao_prioridade = pd.read_sql(
    text(
        """
        SELECT
            regiao,
            uf,
            COUNT(*) AS municipios
        FROM vw_prioridade_estrategica
        GROUP BY
            regiao,
            uf
        ORDER BY
            regiao,
            municipios DESC;
        """
    ),
    engine
)


print(
    "\nDistribuição por região / UF:"
)


display(
    distribuicao_prioridade
)

VALIDAÇÃO — VW_PRIORIDADE_ESTRATEGICA

Registros: 225
Municípios distintos: 225
Duplicatas: 0

Distribuição por região / UF:


,regiao,uf,municipios
0,Centro-Oeste,MT,59
1,Centro-Oeste,MS,13
2,Centro-Oeste,GO,12
3,Sul,RS,73
4,Sul,PR,54
5,Sul,SC,14


In [7]:
# ============================================================
# VIEW — HISTÓRICO AGROAMBIENTAL + PRIORIZAÇÃO
# ============================================================

sql_view_historico = """
CREATE OR REPLACE VIEW vw_historico_priorizacao AS

SELECT
    f.codigo_ibge,
    f.municipio,
    f.uf,
    f.regiao,
    f.ano,
    f.cultura,

    -- ========================================================
    -- PRODUÇÃO AGRÍCOLA
    -- ========================================================

    f.area_plantada_ha,
    f.area_colhida_ha,
    f.quantidade_produzida_t,
    f.rendimento_medio_kg_ha,
    f.valor_producao_mil_reais,

    -- ========================================================
    -- CLIMA
    -- ========================================================

    f.precipitacao_anual_mm,
    f.temperatura_media_anual_c,
    f.umidade_media_anual_pct,
    f.qualidade_climatica_geral,

    -- ========================================================
    -- SOLO E COBERTURA
    -- ========================================================

    f.carbono_solo_t_ha,
    f.pct_cobertura_natural,
    f.pct_agropecuaria,
    f.pct_soja_mapbiomas,

    -- ========================================================
    -- EMISSÕES / CONTEXTO AGROAMBIENTAL
    -- ========================================================

    f.emissao_total_co2e_gwp_ar6_t,
    f.status_dado_seeg,
    f.percentual_conversao_para_soja_pct,
    f.taxa_emissao_co2_t_ha_ano,

    -- ========================================================
    -- RESULTADOS DO MODELO DE PRIORIZAÇÃO
    -- ========================================================

    m.elegivel_cruzamento_priorizacao,
    m.score_relevancia_produtiva,
    m.score_pressao_agroambiental,
    m.quadrante_priorizacao,
    m.faixa_confianca_modelo,

    m.quantidade_cenarios_estrategico,
    m.faixa_robustez_priorizacao,

    m.flag_prioridade_estrategica_base,
    m.flag_prioridade_robusta_3_ou_4_cenarios

FROM fato_agroambiental_anual f

LEFT JOIN mart_priorizacao_municipal m
    ON f.codigo_ibge = m.codigo_ibge;
"""


with engine.begin() as conexao:

    conexao.execute(
        text(
            sql_view_historico
        )
    )


print("=" * 70)
print("VIEW CRIADA")
print("=" * 70)


print(
    "\nvw_historico_priorizacao"
)

VIEW CRIADA

vw_historico_priorizacao


In [8]:
# ============================================================
# VALIDAÇÃO — VW HISTÓRICO PRIORIZAÇÃO
# ============================================================

with engine.connect() as conexao:

    total_historico = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM vw_historico_priorizacao;
                """
            )
        )
        .scalar()
    )


    municipios_historico = (
        conexao.execute(
            text(
                """
                SELECT COUNT(DISTINCT codigo_ibge)
                FROM vw_historico_priorizacao;
                """
            )
        )
        .scalar()
    )


    duplicatas_chave_historico = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        ano,
                        COUNT(*) AS quantidade
                    FROM vw_historico_priorizacao
                    GROUP BY
                        codigo_ibge,
                        ano
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


    sem_priorizacao = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM vw_historico_priorizacao
                WHERE
                    elegivel_cruzamento_priorizacao IS NULL;
                """
            )
        )
        .scalar()
    )


print("=" * 70)
print("VALIDAÇÃO — VW_HISTORICO_PRIORIZACAO")
print("=" * 70)


print(
    "\nRegistros:",
    total_historico
)


print(
    "Municípios:",
    municipios_historico
)


print(
    "Duplicatas codigo_ibge + ano:",
    duplicatas_chave_historico
)


print(
    "Registros sem correspondência no Mart:",
    sem_priorizacao
)


# ------------------------------------------------------------
# Amostra
# ------------------------------------------------------------

amostra_historico = pd.read_sql(
    text(
        """
        SELECT
            codigo_ibge,
            municipio,
            uf,
            ano,
            quantidade_produzida_t,
            precipitacao_anual_mm,
            carbono_solo_t_ha,
            score_relevancia_produtiva,
            score_pressao_agroambiental,
            quadrante_priorizacao,
            flag_prioridade_estrategica_base
        FROM vw_historico_priorizacao
        ORDER BY
            codigo_ibge,
            ano
        LIMIT 12;
        """
    ),
    engine
)


print(
    "\nAmostra:"
)


display(
    amostra_historico
)

VALIDAÇÃO — VW_HISTORICO_PRIORIZACAO

Registros: 8674
Municípios: 1505
Duplicatas codigo_ibge + ano: 0
Registros sem correspondência no Mart: 0

Amostra:


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,precipitacao_anual_mm,carbono_solo_t_ha,score_relevancia_produtiva,score_pressao_agroambiental,quadrante_priorizacao,flag_prioridade_estrategica_base
0,4100103,Abatiá,PR,2019,36784.0,994.780491,55.173093,66.928759,42.682689,Alta relevância + Baixa pressão,0
1,4100103,Abatiá,PR,2020,36435.0,907.189175,55.169301,66.928759,42.682689,Alta relevância + Baixa pressão,0
2,4100103,Abatiá,PR,2021,27727.0,1219.103697,55.039162,66.928759,42.682689,Alta relevância + Baixa pressão,0
3,4100103,Abatiá,PR,2022,37064.0,1501.673064,55.024834,66.928759,42.682689,Alta relevância + Baixa pressão,0
4,4100103,Abatiá,PR,2023,35856.0,1697.612945,55.025820,66.928759,42.682689,Alta relevância + Baixa pressão,0
5,4100103,Abatiá,PR,2024,34560.0,999.928713,55.022820,66.928759,42.682689,Alta relevância + Baixa pressão,0
6,4100202,Adrianópolis,PR,2023,70.0,2018.039145,56.421824,NaN,NaN,Dados insuficientes,0
7,4100202,Adrianópolis,PR,2024,61.0,1653.049405,56.418058,NaN,NaN,Dados insuficientes,0
8,4100301,Agudos do Sul,PR,2019,5365.0,1354.124036,66.305396,50.750536,49.218711,Alta relevância + Baixa pressão,0
9,4100301,Agudos do Sul,PR,2020,7084.0,1185.581568,66.255196,50.750536,49.218711,Alta relevância + Baixa pressão,0


In [9]:
# ============================================================
# VIEW — MUNICÍPIOS PARA POWER BI
# ============================================================

sql_view_powerbi_municipios = """
CREATE OR REPLACE VIEW vw_powerbi_municipios AS

SELECT
    -- ========================================================
    -- IDENTIFICAÇÃO
    -- ========================================================

    codigo_ibge,
    municipio,
    uf,
    regiao,

    -- ========================================================
    -- PRODUÇÃO
    -- ========================================================

    producao_media_t,
    area_colhida_media_ha,
    rendimento_medio_kg_ha,
    rendimento_cv_pct,

    -- ========================================================
    -- CLIMA
    -- ========================================================

    precipitacao_media_mm,
    precipitacao_cv_pct,
    temperatura_media_c,
    temperatura_desvio_padrao_c,
    umidade_media_pct,

    pior_qualidade_climatica,

    -- ========================================================
    -- SOLO / COBERTURA / USO DA TERRA
    -- ========================================================

    carbono_solo_2024_t_ha,
    cobertura_natural_2024_pct,
    soja_mapbiomas_2024_pct,
    agricultura_2024_pct,

    -- ========================================================
    -- EMISSÕES E BRLUC
    -- ========================================================

    seeg_co2e_medio_t,
    percentual_conversao_para_soja_pct,
    taxa_emissao_co2_t_ha_ano,

    -- ========================================================
    -- MODELO DE PRIORIZAÇÃO
    -- ========================================================

    elegivel_cruzamento_priorizacao,

    score_relevancia_produtiva,
    score_pressao_agroambiental,

    quadrante_priorizacao,
    faixa_confianca_modelo,

    -- ========================================================
    -- SENSIBILIDADE / ROBUSTEZ
    -- ========================================================

    quantidade_cenarios_estrategico,
    faixa_robustez_priorizacao,

    flag_prioridade_estrategica_base,
    flag_prioridade_robusta_3_ou_4_cenarios

FROM mart_priorizacao_municipal;
"""


with engine.begin() as conexao:

    conexao.execute(
        text(
            sql_view_powerbi_municipios
        )
    )


print("=" * 70)
print("VIEW CRIADA")
print("=" * 70)


print(
    "\nvw_powerbi_municipios"
)

VIEW CRIADA

vw_powerbi_municipios


In [10]:
# ============================================================
# VALIDAÇÃO — VW_POWERBI_MUNICIPIOS
# ============================================================

with engine.connect() as conexao:

    validacao_powerbi_municipios = (
        conexao.execute(
            text(
                """
                SELECT
                    COUNT(*) AS registros,

                    COUNT(
                        DISTINCT codigo_ibge
                    ) AS municipios,

                    SUM(
                        CASE
                            WHEN elegivel_cruzamento_priorizacao = TRUE
                            THEN 1
                            ELSE 0
                        END
                    ) AS elegiveis,

                    SUM(
                        CASE
                            WHEN flag_prioridade_estrategica_base = TRUE
                            THEN 1
                            ELSE 0
                        END
                    ) AS estrategicos,

                    SUM(
                        CASE
                            WHEN flag_prioridade_robusta_3_ou_4_cenarios = TRUE
                            THEN 1
                            ELSE 0
                        END
                    ) AS robustos

                FROM vw_powerbi_municipios;
                """
            )
        )
        .mappings()
        .one()
    )


print("=" * 70)
print("VALIDAÇÃO — VW_POWERBI_MUNICIPIOS")
print("=" * 70)


print(
    "\nRegistros:",
    validacao_powerbi_municipios[
        "registros"
    ]
)


print(
    "Municípios:",
    validacao_powerbi_municipios[
        "municipios"
    ]
)


print(
    "Elegíveis:",
    validacao_powerbi_municipios[
        "elegiveis"
    ]
)


print(
    "Prioridade estratégica:",
    validacao_powerbi_municipios[
        "estrategicos"
    ]
)


print(
    "Robustos >= 3/4:",
    validacao_powerbi_municipios[
        "robustos"
    ]
)

VALIDAÇÃO — VW_POWERBI_MUNICIPIOS

Registros: 1505
Municípios: 1505
Elegíveis: 1400
Prioridade estratégica: 225
Robustos >= 3/4: 219


In [11]:
# ============================================================
# VIEW — SENSIBILIDADE EM FORMATO LONGO PARA POWER BI
# ============================================================

sql_view_powerbi_sensibilidade = """
CREATE OR REPLACE VIEW vw_powerbi_sensibilidade AS

SELECT
    codigo_ibge,
    municipio,
    uf,
    regiao,

    'base' AS cenario,

    score_relevancia__base AS score_relevancia,
    score_pressao__base AS score_pressao,
    estrategico__base AS estrategico

FROM fato_sensibilidade_priorizacao


UNION ALL


SELECT
    codigo_ibge,
    municipio,
    uf,
    regiao,

    'produtivo' AS cenario,

    score_relevancia__produtivo AS score_relevancia,
    score_pressao__produtivo AS score_pressao,
    estrategico__produtivo AS estrategico

FROM fato_sensibilidade_priorizacao


UNION ALL


SELECT
    codigo_ibge,
    municipio,
    uf,
    regiao,

    'ambiental' AS cenario,

    score_relevancia__ambiental AS score_relevancia,
    score_pressao__ambiental AS score_pressao,
    estrategico__ambiental AS estrategico

FROM fato_sensibilidade_priorizacao


UNION ALL


SELECT
    codigo_ibge,
    municipio,
    uf,
    regiao,

    'climatico' AS cenario,

    score_relevancia__climatico AS score_relevancia,
    score_pressao__climatico AS score_pressao,
    estrategico__climatico AS estrategico

FROM fato_sensibilidade_priorizacao;
"""


with engine.begin() as conexao:

    conexao.execute(
        text(
            sql_view_powerbi_sensibilidade
        )
    )


print("=" * 70)
print("VIEW CRIADA")
print("=" * 70)


print(
    "\nvw_powerbi_sensibilidade"
)

VIEW CRIADA

vw_powerbi_sensibilidade


In [12]:
# ============================================================
# VALIDAÇÃO — VW_POWERBI_SENSIBILIDADE
# ============================================================

with engine.connect() as conexao:

    validacao_sensibilidade = (
        conexao.execute(
            text(
                """
                SELECT
                    COUNT(*) AS registros,

                    COUNT(
                        DISTINCT codigo_ibge
                    ) AS municipios,

                    COUNT(
                        DISTINCT cenario
                    ) AS cenarios

                FROM vw_powerbi_sensibilidade;
                """
            )
        )
        .mappings()
        .one()
    )


    duplicatas_sensibilidade = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        cenario,
                        COUNT(*) AS quantidade
                    FROM vw_powerbi_sensibilidade
                    GROUP BY
                        codigo_ibge,
                        cenario
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


    null_sensibilidade = (
        conexao.execute(
            text(
                """
                SELECT COUNT(*)
                FROM vw_powerbi_sensibilidade
                WHERE
                    codigo_ibge IS NULL
                    OR cenario IS NULL
                    OR score_relevancia IS NULL
                    OR score_pressao IS NULL
                    OR estrategico IS NULL;
                """
            )
        )
        .scalar()
    )


print("=" * 70)
print("VALIDAÇÃO — VW_POWERBI_SENSIBILIDADE")
print("=" * 70)


print(
    "\nRegistros:",
    validacao_sensibilidade[
        "registros"
    ]
)


print(
    "Municípios:",
    validacao_sensibilidade[
        "municipios"
    ]
)


print(
    "Cenários:",
    validacao_sensibilidade[
        "cenarios"
    ]
)


print(
    "Duplicatas codigo_ibge + cenario:",
    duplicatas_sensibilidade
)


print(
    "Registros com NULL:",
    null_sensibilidade
)

VALIDAÇÃO — VW_POWERBI_SENSIBILIDADE

Registros: 5600
Municípios: 1400
Cenários: 4
Duplicatas codigo_ibge + cenario: 0
Registros com NULL: 0


In [13]:
# ============================================================
# DISTRIBUIÇÃO — CENÁRIOS DE SENSIBILIDADE
# ============================================================

distribuicao_cenarios_powerbi = pd.read_sql(
    text(
        """
        SELECT
            cenario,

            COUNT(*) AS municipios,

            SUM(
                CASE
                    WHEN estrategico = TRUE
                    THEN 1
                    ELSE 0
                END
            ) AS estrategicos,

            ROUND(
                AVG(score_relevancia),
                4
            ) AS score_relevancia_medio,

            ROUND(
                AVG(score_pressao),
                4
            ) AS score_pressao_medio

        FROM vw_powerbi_sensibilidade

        GROUP BY
            cenario

        ORDER BY
            CASE cenario
                WHEN 'base' THEN 1
                WHEN 'produtivo' THEN 2
                WHEN 'ambiental' THEN 3
                WHEN 'climatico' THEN 4
                ELSE 5
            END;
        """
    ),
    engine
)


print("=" * 70)
print("CENÁRIOS — VW_POWERBI_SENSIBILIDADE")
print("=" * 70)


display(
    distribuicao_cenarios_powerbi
)

CENÁRIOS — VW_POWERBI_SENSIBILIDADE


,cenario,municipios,estrategicos,score_relevancia_medio,score_pressao_medio
0,base,1400,225.0,50.0,50.2657
1,produtivo,1400,250.0,50.0,50.2657
2,ambiental,1400,242.0,50.0,50.2657
3,climatico,1400,247.0,50.0,50.2657


In [14]:
# ============================================================
# INVENTÁRIO FINAL — TABELAS E VIEWS MYSQL
# ============================================================

objetos_mysql = pd.read_sql(
    text(
        """
        SELECT
            TABLE_NAME AS objeto,
            TABLE_TYPE AS tipo
        FROM information_schema.TABLES
        WHERE
            TABLE_SCHEMA = 'agroesg_analytics'
        ORDER BY
            TABLE_TYPE,
            TABLE_NAME;
        """
    ),
    engine
)


print("=" * 70)
print("OBJETOS — AGROESG_ANALYTICS")
print("=" * 70)


display(
    objetos_mysql
)

OBJETOS — AGROESG_ANALYTICS


,objeto,tipo
0,dim_cenario_sensibilidade,BASE TABLE
1,fato_agroambiental_anual,BASE TABLE
2,fato_sensibilidade_priorizacao,BASE TABLE
3,mart_priorizacao_municipal,BASE TABLE
4,vw_historico_priorizacao,VIEW
5,vw_powerbi_municipios,VIEW
6,vw_powerbi_sensibilidade,VIEW
7,vw_prioridade_estrategica,VIEW
